# Advanced Problems with Solutions: Closure Applications

Topic: running averages, closure-based state, class-to-closure conversions, timers, callable objects, and practical closure patterns.


## Problem 1: Basic Closure Averager

Create a closure-based averager. Each call should accept one number and return the average of all numbers seen so far.

In [1]:
def averager():
    numbers = []

    def add(number):
        numbers.append(number)
        return sum(numbers) / len(numbers)

    return add

avg = averager()

print(avg(10))
print(avg(20))
print(avg(30))

10.0
15.0
20.0


### Solution Explanation

`numbers` is captured by the inner function. The list remains alive after `averager()` returns because the returned function references it.

Since the list is mutated but not reassigned, `nonlocal` is not required.

## Problem 2: Efficient Running Average

Improve the previous solution so it does not store every number. Store only `total` and `count`.

In [2]:
def efficient_averager():
    total = 0
    count = 0

    def add(value):
        nonlocal total, count
        total += value
        count += 1
        return total / count

    return add

avg = efficient_averager()

print(avg(10))
print(avg(20))
print(avg(30))

10.0
15.0
20.0


### Solution Explanation

`total` and `count` are reassigned inside `add`, so they must be declared `nonlocal`.

This version is more memory-efficient because it stores only two numbers instead of the full history.

## Problem 3: Compare Class and Closure Averagers

Implement the same running average behavior using both a class and a closure.

In [3]:
class Averager:
    def __init__(self):
        self.total = 0
        self.count = 0

    def add(self, value):
        self.total += value
        self.count += 1
        return self.total / self.count


def make_averager():
    total = 0
    count = 0

    def add(value):
        nonlocal total, count
        total += value
        count += 1
        return total / count

    return add

class_avg = Averager()
closure_avg = make_averager()

for value in [10, 20, 30]:
    print(class_avg.add(value), closure_avg(value))

10.0 10.0
15.0 15.0
20.0 20.0


### Solution Explanation

The class stores state in instance attributes.

The closure stores state in captured variables.

Both approaches are valid. Closures are often useful when the object has one main behavior.

## Problem 4: Callable Class vs Closure Timer

Create a timer using a callable class and then rewrite it using a closure.

In [4]:
from time import perf_counter, sleep

class Timer:
    def __init__(self):
        self.start = perf_counter()

    def __call__(self):
        return perf_counter() - self.start


def timer():
    start = perf_counter()

    def elapsed():
        return perf_counter() - start

    return elapsed

t1 = Timer()
t2 = timer()

sleep(0.1)

print(t1())
print(t2())

0.10117239970713854
0.10144240036606789


### Solution Explanation

`Timer.__call__` makes the instance callable.

The closure version captures `start`. Since `start` is only read, `nonlocal` is not needed.

## Problem 5: Independent Timer Instances

Create two closure timers at different times and prove they have independent captured start times.

In [5]:
from time import perf_counter, sleep

def timer():
    start = perf_counter()

    def elapsed():
        return perf_counter() - start

    return elapsed

a = timer()
sleep(0.1)
b = timer()
sleep(0.1)

print(a())
print(b())

print(a.__closure__[0] is b.__closure__[0])

0.21183069981634617
0.10935840010643005
False


### Solution Explanation

Every call to `timer()` creates a new local scope and a new closure cell for `start`.

Therefore, `a` and `b` track different start times.

## Problem 6: Resettable Closure Timer

Create a timer closure with two operations:

- `elapsed()`
- `reset()`

In [6]:
from time import perf_counter, sleep

def resettable_timer():
    start = perf_counter()

    def elapsed():
        return perf_counter() - start

    def reset():
        nonlocal start
        start = perf_counter()

    return elapsed, reset

elapsed, reset = resettable_timer()

sleep(0.1)
print(elapsed())

reset()
sleep(0.1)
print(elapsed())

0.10601869970560074
0.10061600059270859


### Solution Explanation

`elapsed` only reads `start`.

`reset` reassigns `start`, so it requires `nonlocal start`.

## Problem 7: Averager with Reset

Create a running averager that supports:

- adding values
- getting the current average
- resetting the state

In [7]:
def make_resettable_averager():
    total = 0
    count = 0

    def add(value):
        nonlocal total, count
        total += value
        count += 1
        return total / count

    def average():
        if count == 0:
            return None
        return total / count

    def reset():
        nonlocal total, count
        total = 0
        count = 0

    return add, average, reset

add, average, reset = make_resettable_averager()

print(add(10))
print(add(20))
print(average())
reset()
print(average())

10.0
15.0
15.0
None


### Solution Explanation

`add`, `average`, and `reset` share the same captured state.

`average` does not need `nonlocal` because it only reads the state.

## Problem 8: Windowed Moving Average

Create a closure that keeps only the last `n` values and returns their average.

In [8]:
from collections import deque

def moving_average(window_size):
    values = deque(maxlen=window_size)

    def add(value):
        values.append(value)
        return sum(values) / len(values)

    return add

ma = moving_average(3)

print(ma(10))
print(ma(20))
print(ma(30))
print(ma(40))
print(ma(50))

10.0
15.0
20.0
30.0
40.0


### Solution Explanation

`deque(maxlen=window_size)` automatically discards the oldest value when the window is full.

The closure captures the deque and updates it on each call.

## Problem 9: Efficient Moving Average

Improve the moving average so it does not call `sum(values)` every time.

In [9]:
from collections import deque

def efficient_moving_average(window_size):
    values = deque()
    total = 0

    def add(value):
        nonlocal total

        values.append(value)
        total += value

        if len(values) > window_size:
            removed = values.popleft()
            total -= removed

        return total / len(values)

    return add

ma = efficient_moving_average(3)

print(ma(10))
print(ma(20))
print(ma(30))
print(ma(40))
print(ma(50))

10.0
15.0
20.0
30.0
40.0


### Solution Explanation

The closure stores a running `total` and subtracts the removed value when the window exceeds the allowed size.

This avoids recalculating the sum from scratch on every call.

## Problem 10: Closure-Based Min/Max Tracker

Create a closure that accepts numbers and returns a tuple containing:

- current minimum
- current maximum
- current average

In [10]:
def stats_tracker():
    total = 0
    count = 0
    minimum = None
    maximum = None

    def add(value):
        nonlocal total, count, minimum, maximum

        total += value
        count += 1

        minimum = value if minimum is None else min(minimum, value)
        maximum = value if maximum is None else max(maximum, value)

        return minimum, maximum, total / count

    return add

stats = stats_tracker()

print(stats(10))
print(stats(5))
print(stats(20))
print(stats(15))

(10, 10, 10.0)
(5, 10, 7.5)
(5, 20, 11.666666666666666)
(5, 20, 12.5)


### Solution Explanation

All four state variables are reassigned inside `add`, so all require `nonlocal`.

This is a practical example of closures as lightweight state machines.

## Problem 11: Running Product Tracker

Create a closure that returns the running product of all numbers added so far.

In [11]:
def running_product():
    product = 1
    count = 0

    def add(value):
        nonlocal product, count
        product *= value
        count += 1
        return product

    return add

rp = running_product()

print(rp(2))
print(rp(3))
print(rp(4))

2
6
24


### Solution Explanation

The closure keeps state between calls.

Unlike the averager, only one running variable (`product`) is necessary.

## Problem 12: Closure-Based Event Counter

Create a closure that counts how many times it has been called.

In [12]:
def call_counter():
    calls = 0

    def counter():
        nonlocal calls
        calls += 1
        return calls

    return counter

counter = call_counter()

print(counter())
print(counter())
print(counter())

1
2
3


### Solution Explanation

`calls` persists across function invocations because it is captured by the closure.

`nonlocal` is necessary because the value is reassigned.

## Problem 13: Threshold Alarm

Create a closure that triggers an alert after a running total exceeds a threshold.

In [13]:
def threshold_alarm(limit):
    total = 0

    def add(value):
        nonlocal total
        total += value

        if total >= limit:
            return f'ALERT: threshold {limit} reached'

        return total

    return add

alarm = threshold_alarm(100)

print(alarm(20))
print(alarm(30))
print(alarm(50))

20
50
ALERT: threshold 100 reached


### Solution Explanation

Closures are useful for stateful event monitoring.

The threshold value remains fixed while the total evolves over time.

## Problem 14: Stopwatch with Lap Times

Create a closure stopwatch that supports lap timing.

In [14]:
from time import perf_counter, sleep

def stopwatch():
    start = perf_counter()
    last_lap = start

    def lap():
        nonlocal last_lap

        now = perf_counter()
        elapsed = now - last_lap
        last_lap = now
        return elapsed

    def total():
        return perf_counter() - start

    return lap, total

lap, total = stopwatch()

sleep(0.1)
print(lap())

sleep(0.1)
print(lap())

print(total())

0.10069520026445389
0.10176619980484247
0.20262350048869848


### Solution Explanation

`last_lap` changes after each lap and therefore needs `nonlocal`.

`start` remains fixed and is only read.

## Problem 15: Closure-Based Rate Limiter

Create a closure that only allows a function to be called a maximum number of times.

In [15]:
def limit_calls(fn, max_calls):
    calls = 0

    def wrapper(*args, **kwargs):
        nonlocal calls

        if calls >= max_calls:
            raise RuntimeError('Call limit exceeded')

        calls += 1
        return fn(*args, **kwargs)

    return wrapper

def greet(name):
    return f'Hello, {name}!'

limited_greet = limit_calls(greet, 2)

print(limited_greet('Alice'))
print(limited_greet('Bob'))

try:
    print(limited_greet('Charlie'))
except RuntimeError as ex:
    print(ex)

Hello, Alice!
Hello, Bob!
Call limit exceeded


### Solution Explanation

The closure stores state (`calls`) while still behaving like a regular function wrapper.

This is a common decorator-style use case.

## Problem 16: Benchmark Wrapper

Create a closure-based benchmark wrapper that measures how long a function takes to run.

In [16]:
from time import perf_counter, sleep

def benchmark(fn):
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = fn(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper

@benchmark
def slow_task(seconds):
    sleep(seconds)
    return 'done'

result, elapsed = slow_task(0.1)

print(result)
print(elapsed)

done
0.10012950003147125


### Solution Explanation

The wrapper closure adds timing functionality while preserving the original function interface.

## Problem 17: Closure-Based Configuration Factory

Create a configurable formatter closure.

In [17]:
def formatter(prefix='', suffix=''):
    def format_text(text):
        return f'{prefix}{text}{suffix}'

    return format_text

bold = formatter('**', '**')
warning = formatter('[WARNING] ', '')

print(bold('important'))
print(warning('disk space low'))

**important**
[WARNING] disk space low


### Solution Explanation

Closures can freeze configuration into a callable function.

`prefix` and `suffix` remain available long after the outer function finishes.

## Problem 18: Stateful Incrementer Factory

Create a configurable incrementer.

In [18]:
def incrementer(step):
    current = 0

    def increment():
        nonlocal current
        current += step
        return current

    return increment

inc_2 = incrementer(2)
inc_5 = incrementer(5)

print(inc_2())
print(inc_2())
print(inc_5())
print(inc_5())

2
4
5
10


### Solution Explanation

Each incrementer instance gets its own independent closure state.

## Problem 19: Pause/Resume Timer

Create a timer that supports pausing and resuming.

In [19]:
from time import perf_counter, sleep

def pauseable_timer():
    start = perf_counter()
    paused_time = 0
    pause_start = None

    def elapsed():
        current = perf_counter()
        paused = 0

        if pause_start is not None:
            paused = current - pause_start

        return current - start - paused_time - paused

    def pause():
        nonlocal pause_start
        if pause_start is None:
            pause_start = perf_counter()

    def resume():
        nonlocal pause_start, paused_time

        if pause_start is not None:
            paused_time += perf_counter() - pause_start
            pause_start = None

    return elapsed, pause, resume

elapsed, pause, resume = pauseable_timer()

sleep(0.1)
pause()

sleep(0.2)
resume()

sleep(0.1)
print(elapsed())

0.2014586003497243


### Solution Explanation

The timer tracks total paused duration and subtracts it from elapsed time.

This demonstrates a more advanced multi-state closure.

## Problem 20: Final Challenge — Closure-Based Task Scheduler

Create a mini task scheduler using closures.

In [20]:
def task_scheduler():
    tasks = []

    def add_task(task):
        tasks.append(task)

    def run_tasks():
        return [task() for task in tasks]

    def list_tasks():
        return len(tasks)

    return add_task, run_tasks, list_tasks

add_task, run_tasks, list_tasks = task_scheduler()

add_task(lambda: 'Task 1 complete')
add_task(lambda: 'Task 2 complete')

print(list_tasks())
print(run_tasks())

2
['Task 1 complete', 'Task 2 complete']


### Final Challenge Explanation

The closure stores private state (`tasks`) without exposing direct access.

This pattern resembles lightweight object-oriented programming using functions.